# stAge quickstart: predicting transcriptomic age on one small dataset

This walks through the core pipeline end to end on `brain3g` (GEO series `GSE212903`, 4 samples — the smallest dataset encountered in this codebase), from raw counts to a spatial transcriptomic-age (tAge) map.

Steps: metapixel clustering (SpatialGroup) → gene filtering → normalization → elastic-net clock prediction → propagation back to pixel level → a spatial plot.

**Before running:** `conda env create -f ../environment.yml && conda activate stage-release`, then `python ../data/download.py --dataset brain3g` to fetch the raw data, and make sure you have a copy of the trained clock `.pkl` files and an NCBI `*.gene_info` reference file (see `../README.md` — these are the paper's trained models / a standard NCBI reference file, not produced by this notebook).

In [2]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import scanpy as sc
import matplotlib.pyplot as plt

from stage.metapixels import non_overlapping_MPs
from stage.pipeline import full_nonoverlap_mp_pipeline
from stage.resolution_search import optimal_resolution_search

## 1. Load the raw data

Fill in paths to match wherever `data/download.py` (or your own conversion via `../R/rds_to_h5ad.R` / `../v_pipeline`-equivalent `convert_to_h5ad` step, if starting from a non-h5ad raw format) put the 4 brain3g samples. Each `AnnData` needs `.obs['total_counts']`, `.obs['cell_type']`, `.obsm['spatial']`, and `.layers['raw_count']` populated — see `stage/metapixels.py` docstrings for exactly what each pipeline stage expects.

In [ ]:
RAW_DIR = '../data/raw/brain3g'  # adjust to your actual download location

anndata_dict = {
    f: sc.read_h5ad(os.path.join(RAW_DIR, f))
    for f in os.listdir(RAW_DIR) if f.endswith('.h5ad')
}
print(f'Loaded {len(anndata_dict)} samples: {list(anndata_dict)}')

## 2. (Optional) Optimal resolution search

Sweeps Leiden resolutions and scores each by the composite score `S = 0.4·norm(|t|) + 0.6·norm(|Cohen's d|)` (young vs. old tAge separation). This re-runs the full prediction pipeline once per resolution, so it's the slowest step — skip it and just pick a resolution (e.g. `res=2`) for a fast first run.

In [ ]:
CLOCK_ROOT = '/path/to/tAge_clocks'  # the paper's trained clock directory (see README.md)
GENE_INFO = '/path/to/Mus_musculus.gene_info'  # standard NCBI gene_info reference

def _pred_pipeline(assembled_adatas, res, control_file_pattern, mp_coverage_threshold,
                    lower_res, save_plot, save_result, clock_folder, save_dir, tag):
    return full_nonoverlap_mp_pipeline(
        assembled_adatas, ncbi_reference_path=GENE_INFO, clock_root=CLOCK_ROOT,
        res=res, lower_res=lower_res, control_file_pattern=control_file_pattern,
        mp_coverage_threshold=mp_coverage_threshold, save_plot=save_plot,
        save_result=save_result, clock_folder=clock_folder, save_dir=save_dir, tag=tag,
    )

ors_result = optimal_resolution_search(
    anndata_dict, ipynb_dir='.', pred_pipeline=_pred_pipeline,
    control_file_pattern='Young', tolerance=0.1,  # 10% tolerance, matches the paper's reported ORS methodology
    clock_dirs={'orig': os.path.join(CLOCK_ROOT, 'EN differential models 4.6')},
)
ors_result

## 3. Run the prediction pipeline at a chosen resolution

In [ ]:
preds_per_file = full_nonoverlap_mp_pipeline(
    anndata_dict,
    ncbi_reference_path=GENE_INFO,
    clock_root=CLOCK_ROOT,
    clock_folder='tAge_clocks/EN differential models 4.6',
    res=2,                       # or ors_result's best resolution from step 2
    lower_res=False,             # False: propagate predictions back to pixel level
    control_file_pattern='Young',
    mp_coverage_threshold=150_000,
    save_result=False,
)
list(preds_per_file)

## 4. Plot spatial tAge for one sample

In [ ]:
sample = next(iter(preds_per_file))
adata = preds_per_file[sample]

fig, ax = plt.subplots(figsize=(5, 5))
sc.pl.embedding(adata, basis='spatial', color='tAge_SM', ax=ax, show=False, title=f'{sample}: tAge (scaled clock)')
plt.show()

## Next steps

- `../analyses/` has one script per paper figure, each importing from `stage/` the same way this notebook does.
- `../INVENTORY.md` documents where every analysis's source code came from and any open gaps/caveats.
- Getis-Ord Gi* hotspot calling (`stage.hotspots`), composition-independence (`stage.composition`), and meta-analysis/GSEA (`stage.meta_analysis`, `stage.gsea`) all build on the `tAge_SM`/`tAge_YM` columns this notebook just produced.